# 피드백 모델 만들기: 규칙으로 고른 짝 + DPO

**2장의 피드백 모델(`load_story_llm("feedback")`)을 만든 노트북입니다.** 실행하지 않아도 책을 따라가는 데 문제가 없습니다.

파인튜닝 모델은 무엇을 물어도 이야기를 지어냅니다. 여기서는 사람이 하나하나 고르는 대신 **규칙으로 더 좋은 답을 고른 짝(선호 데이터)**을 만들고, 그 짝으로 파인튜닝 모델을 한 번 더 학습시킵니다(DPO).

- 규칙: 요청이 이야기 부탁이면 **이야기**가 더 좋은 답, 이야기가 아닌 질문이면 **모른다는 답**이 더 좋은 답
- 이야기가 아닌 질문 460개: 더 좋은 답 = 모른다는 답, 덜 좋은 답 = 파인튜닝 모델이 지어낸 이야기
- 이야기 부탁 460개(요청-이야기 데이터에서 뽑음): 더 좋은 답 = 데이터의 이야기, 덜 좋은 답 = 모른다는 답
- 시험용으로 "대한민국의 수도는 어디야?"와 '대한민국'·'한국'이 들어간 질문은 학습 데이터에서 뺍니다.

CPU에서 짝 만들기 약 20분, 학습 약 7분이 걸립니다(GPU는 더 빠름).

In [ ]:
%pip install -q git+https://github.com/yhlee1627/modu-llm-v2 datasets

## 1. 설정과 질문 목록

In [ ]:
import json
import random
import time

import pandas as pd
import torch
from torch.nn import functional as F
from datasets import load_dataset
from huggingface_hub import hf_hub_download

from modu_llm import ASSET_REPO, SFT_REPO, Vocab, generate, load_checkpoint, save_checkpoint
from modu_llm.sft import format_pair

REFUSE = "그 질문의 답은 몰라요. 저는 이야기를 쓰는 법만 배웠어요. 이야기를 부탁해 주면 재미있는 이야기를 들려줄게요."
BLOCK = 384


In [ ]:
COUNTRIES = ["일본", "중국", "미국", "영국", "프랑스", "독일", "이탈리아", "스페인", "캐나다", "호주", "브라질", "인도",
             "러시아", "이집트", "태국", "베트남", "멕시코", "아르헨티나", "그리스", "튀르키예", "스웨덴", "노르웨이",
             "핀란드", "덴마크", "네덜란드", "벨기에", "스위스", "오스트리아", "폴란드", "포르투갈", "필리핀",
             "인도네시아", "몽골", "케냐", "칠레", "페루", "뉴질랜드", "아일랜드", "헝가리", "체코"]
WORDS = ["사과", "학교", "고양이", "자동차", "컴퓨터", "바다", "하늘", "친구", "책", "연필", "의자", "우유", "나무", "기차",
         "비행기", "강아지", "꽃", "물", "엄마", "아빠", "선생님", "병원", "공원", "시계", "가방", "모자", "신발", "창문",
         "구름", "별", "달", "해", "바람", "눈", "비", "숲", "산", "강", "다리", "도서관"]
HAND = """오늘 날씨 어때?
지금 몇 시야?
내일 비 와?
하늘은 왜 파란색이야?
지구에서 가장 높은 산은 어디야?
물은 몇 도에서 끓어?
한글은 누가 만들었어?
피자 만드는 법 알려 줘
파이썬이 뭐야?
공부를 잘하는 방법이 뭐야?
달까지 거리는 얼마나 돼?
태양계에서 가장 큰 행성은 뭐야?
1년은 며칠이야?
고래는 물고기야?
비행기는 어떻게 날아?
가장 빠른 동물은 뭐야?
세종대왕은 누구야?
라면 맛있게 끓이는 법은?
감기에 걸리면 어떻게 해?
축구는 몇 명이 해?
주말에 뭐 하면 좋을까?
영어 공부는 어떻게 해?
오늘 무슨 요일이야?
환율이 얼마야?
서울에서 부산까지 얼마나 걸려?
지구는 왜 돌아?
무지개는 왜 생겨?
바닷물은 왜 짜?
겨울에는 왜 추워?
사람의 뼈는 몇 개야?
심장은 하루에 몇 번 뛰어?
빛은 얼마나 빨라?
컴퓨터는 어떻게 켜?
비밀번호를 잊어버렸어. 어떻게 해?
배가 아파. 무슨 약을 먹어야 해?
가장 큰 바다는 어디야?
가장 긴 강은 어디야?
피아노는 건반이 몇 개야?
올림픽은 몇 년마다 열려?
우리 동네 맛집 추천해 줘
요즘 인기 있는 노래가 뭐야?
내 생일은 언제야?
너는 몇 살이야?
너는 누가 만들었어?
오늘 뉴스 알려 줘
주식은 언제 사야 해?
수학 숙제 좀 풀어 줘
이 문장을 영어로 번역해 줘
편지 쓰는 법 알려 줘
자전거 타는 법 알려 줘
김치는 어떻게 만들어?
시험 공부 계획 세워 줘
잠이 안 와. 어떻게 해?
지구에는 나라가 몇 개 있어?
전기는 어디서 와?
화산은 왜 터져?
지진은 왜 일어나?
공룡은 왜 사라졌어?
개미는 다리가 몇 개야?
거미는 곤충이야?
바나나는 무슨 색이야?
토마토는 과일이야 채소야?
가장 작은 나라는 어디야?
에베레스트 산은 몇 미터야?
일주일은 몇 시간이야?
백 곱하기 백은 얼마야?
소수가 뭐야?
분수 더하기는 어떻게 해?
삼각형 넓이는 어떻게 구해?
원주율은 얼마야?
지금 기온이 몇 도야?
여름 방학은 언제 시작해?
버스는 몇 시에 와?
택시 부르는 법 알려 줘
기차표는 어디서 사?
인터넷이 안 돼. 왜 그래?
휴대폰 배터리가 빨리 닳아. 왜 그래?
가장 오래된 나무는 몇 살이야?
사막에는 왜 비가 안 와?
북극에는 누가 살아?
펭귄은 날 수 있어?
낙타 혹에는 뭐가 들었어?
꿀벌은 꿀을 어떻게 만들어?
비타민은 왜 먹어?
하루에 물을 얼마나 마셔야 해?
운동은 얼마나 해야 해?
좋은 이름 하나 지어 줘
내 키는 몇이야?
점심 메뉴 골라 줘
이번 주 날씨 알려 줘
가장 유명한 그림은 뭐야?
베토벤은 누구야?
지구는 몇 살이야?
우주는 얼마나 커?
블랙홀이 뭐야?
로봇은 어떻게 움직여?
인공지능이 뭐야?
엘엘엠이 뭐야?
코딩은 어떻게 배워?
엑셀에서 합계 구하는 법 알려 줘""".split("\n")


def eul(w):
    c = w[-1]
    return "을" if "가" <= c <= "힣" and (ord(c) - 0xAC00) % 28 else "를"


def questions(rng):
    qs = set(HAND)
    for c in COUNTRIES:
        qs.update([f"{c}의 수도는 어디야?", f"{c}의 수도가 어디인지 알려 줘", f"{c}에서는 어떤 말을 써?",
                   f"{c}의 인구는 몇 명이야?"])
    for w in WORDS:
        qs.update([f"{w}{eul(w)} 영어로 뭐라고 해?", f"{w}의 뜻이 뭐야?"])
    for _ in range(120):
        a, b = rng.randint(1, 99), rng.randint(1, 99)
        qs.add(rng.choice([f"{a} 더하기 {b}는 얼마야?", f"{a} 빼기 {b}는 얼마야?", f"{a} 곱하기 {b}는 뭐야?"]))
    qs = sorted(q for q in qs if "대한민국" not in q and "한국" not in q)
    rng.shuffle(qs)
    return qs


## 2. 선호 데이터 만들기

질문마다 파인튜닝 모델이 지어낸 이야기를 '덜 좋은 답'으로 모읍니다. 이야기가 384글자(문맥 길이)를 넘으면 마지막 마침표에서 자릅니다.

In [ ]:
def make_data():
    rng = random.Random(0)
    torch.manual_seed(0)
    vocab = Vocab.build()
    sft, _, _, _ = load_checkpoint(hf_hub_download(ASSET_REPO, "sft.pt"), "cpu")
    qs = questions(rng)
    df = load_dataset(SFT_REPO, split="train").to_pandas()
    rows = []
    t0 = time.time()
    for i, q in enumerate(qs):
        made = generate(sft, vocab, format_pair(q), max_new_tokens=200, temperature=0.8)
        rows.append({"request": q, "kind": "question", "chosen": REFUSE, "rejected": made})
        if i % 50 == 0:
            print(i, len(qs), f"{time.time() - t0:.0f}s", q, "→", made[:40].replace("\n", " "), flush=True)
    idx = rng.sample(range(len(df)), len(qs))
    for j in idx:
        r = df.iloc[j]
        room = BLOCK - len(format_pair(r["request"])) - 2
        story = r["story"]
        if len(story) > room:                      # 너무 길면 마지막 마침표에서 자름
            cut = story[:room]
            story = cut[:cut.rfind(".") + 1] if "." in cut else cut
        rows.append({"request": r["request"], "kind": "story", "chosen": story, "rejected": REFUSE})
    rng.shuffle(rows)
    unk = set()
    for r in rows:
        for k in ("request", "chosen", "rejected"):
            unk.update(vocab.unknown_chars(r[k]))
    print("사전에 없는 글자:", unk)
    pd.DataFrame(rows).to_parquet("preference.parquet")
    print("저장:", len(rows), "짝 (질문", len(qs), "/ 이야기", len(qs), ")")

make_data()

## 3. DPO로 학습하기

- 기준 모델: 파인튜닝 모델(학습하지 않음). 학습할 모델도 파인튜닝 모델에서 시작합니다.
- DPO 손실: 더 좋은 답의 확률은 기준 모델보다 올리고, 덜 좋은 답의 확률은 내리도록 합니다(`beta=0.1`).
- DPO만으로 학습하면 덜 좋은 답(지어낸 이야기)을 밀어내기만 하고 모른다는 답은 거의 나오지 않았습니다. 그래서 더 좋은 답의 글자당 손실도 함께 줄이도록 더했습니다(`alpha=1.0`, Llama 3 등에서도 쓰는 방법).
- 짝 16개씩, 데이터를 세 바퀴(174번) 학습합니다.

In [ ]:
def seq(vocab, request, answer, end=True):
    """프롬프트와 답의 번호, 그리고 답 부분 표시(손실을 셀 자리)."""
    p = vocab.encode(format_pair(request))
    a = vocab.encode(answer) + ([vocab.end_id] if end else [])
    ids = (p + a)[:BLOCK + 1]
    mask = [0] * len(p) + [1] * len(a)
    return ids, mask[:len(ids)]


def batch_logps(model, items):
    """items: [(ids, mask)]. 답 부분 글자들의 로그 확률 합."""
    T = max(len(i) for i, _ in items)
    x = torch.zeros(len(items), T - 1, dtype=torch.long)
    y = torch.zeros(len(items), T - 1, dtype=torch.long)
    m = torch.zeros(len(items), T - 1)
    for b, (ids, mask) in enumerate(items):
        n = len(ids) - 1
        x[b, :n] = torch.tensor(ids[:-1])
        y[b, :n] = torch.tensor(ids[1:])
        m[b, :n] = torch.tensor(mask[1:], dtype=torch.float)
    logits, _ = model(x)
    lp = torch.gather(F.log_softmax(logits.float(), -1), 2, y.unsqueeze(-1)).squeeze(-1)
    return (lp * m).sum(-1), m.sum(-1)


def train(beta=0.1, lr=5e-5, epochs=3, bs=16, alpha=1.0):
    torch.manual_seed(0)
    vocab = Vocab.build()
    path = hf_hub_download(ASSET_REPO, "sft.pt")
    policy, _, sft_step, history = load_checkpoint(path, "cpu")
    ref, _, _, _ = load_checkpoint(path, "cpu")
    ref.eval()
    df = pd.read_parquet("preference.parquet")
    data = []
    for r in df.itertuples():
        data.append((seq(vocab, r.request, r.chosen), seq(vocab, r.request, r.rejected)))
    # 기준 모델(파인튜닝 모델)의 로그 확률은 미리 한 번만 계산
    with torch.no_grad():
        ref_c, ref_r = [], []
        for i in range(0, len(data), 32):
            ch = data[i:i + 32]
            ref_c.append(batch_logps(ref, [c for c, _ in ch])[0])
            ref_r.append(batch_logps(ref, [r for _, r in ch])[0])
        ref_c, ref_r = torch.cat(ref_c), torch.cat(ref_r)
    opt = torch.optim.AdamW(policy.parameters(), lr=lr, weight_decay=0.0)
    policy.train()
    step = 0
    log = []
    t0 = time.time()
    order = list(range(len(data)))
    rng = random.Random(0)
    for ep in range(epochs):
        rng.shuffle(order)
        for i in range(0, len(order), bs):
            ids = order[i:i + bs]
            pc, nc = batch_logps(policy, [data[j][0] for j in ids])
            pr, _ = batch_logps(policy, [data[j][1] for j in ids])
            margin = beta * ((pc - ref_c[ids]) - (pr - ref_r[ids]))
            dpo = -F.logsigmoid(margin).mean()
            nll = (-pc / nc).mean()                  # 더 좋은 답의 글자당 손실(DPO만으로는 더 좋은 답이 잘 나오지 않아 더함)
            loss = dpo + alpha * nll
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
            opt.step()
            step += 1
            acc = (margin > 0).float().mean().item()
            log.append({"step": step, "loss": loss.item(), "dpo": dpo.item(), "nll": nll.item(), "acc": acc})
            if step % 10 == 0:
                print(f"번 {step:4d} | DPO {dpo.item():.3f} | 좋은 답 손실 {nll.item():.3f} | 맞힌 짝 {acc:.2f} | {time.time() - t0:.0f}s", flush=True)
    policy.eval()
    save_checkpoint("feedback.pt", policy, None, sft_step, history)
    json.dump({"beta": beta, "lr": lr, "epochs": epochs, "batch": bs, "alpha_nll": alpha, "pairs": len(data), "steps": step,
               "log": log}, open("dpo_log.json", "w"), ensure_ascii=False)
    print("저장: feedback.pt", step, "번")

train()

## 4. 확인하기

학습 데이터에 없던 질문과 이야기 부탁을 세 번씩 넣어 봅니다. 마지막 줄은 확인용 요청-이야기 데이터의 손실로, 이야기 실력이 떨어지지 않았는지 봅니다.

In [ ]:
def evaluate():
    vocab = Vocab.build()
    sft, _, _, _ = load_checkpoint(hf_hub_download(ASSET_REPO, "sft.pt"), "cpu")
    fb, _, _, _ = load_checkpoint("feedback.pt", "cpu")
    tests_q = ["대한민국의 수도는 어디야?", "3 더하기 4는 몇이야?", "오늘 저녁 메뉴 추천해 줘", "바다는 왜 넓어?",
               "토끼는 무엇을 먹어?", "구구단 외우는 법 알려 줘", "프랑스 사람들은 무엇을 먹어?", "내일 학교 가?"]
    tests_s = ["토끼가 나오는 짧은 이야기를 써 줘", "곰돌이가 나오는 이야기를 들려줘", "숲속에서 토끼 보리가 나오는 이야기를 들려줘",
               "강아지 뭉치가 나오는 이야기를 만들어 줘", "할머니와 손녀가 나오는 이야기를 써 줘"]
    for name, m in [("파인튜닝", sft), ("피드백", fb)]:
        for label, tests in [("질문", tests_q), ("이야기", tests_s)]:
            refused = 0
            for t in tests:
                for s in range(3):
                    torch.manual_seed(s)
                    out = generate(m, vocab, format_pair(t), max_new_tokens=120)
                    refused += out.startswith("그 질문의 답은 몰라요")
                    if s == 0:
                        print(f"[{name}/{label}] {t} → {out[:70]!r}")
            print(f"== {name} {label}: 모른다는 답 {refused}/{len(tests) * 3}")
    # 이야기 실력이 떨어지지 않았는지: 확인용 요청-이야기 손실
    val = load_dataset(SFT_REPO, split="validation").to_pandas().iloc[:200]
    for name, m in [("파인튜닝", sft), ("피드백", fb)]:
        tot, n = 0.0, 0
        with torch.no_grad():
            for r in val.itertuples():
                ids = vocab.encode(format_pair(r.request, r.story))[:BLOCK + 1] + []
                x = torch.tensor([ids[:-1]]); y = torch.tensor([ids[1:]])
                _, loss = m(x, y)
                tot += loss.item(); n += 1
        print(f"{name} 확인용 이야기 손실: {tot / n:.3f}")

evaluate()

## 5. 올리기

`feedback.pt`를 허깅페이스 `JosephLee/modu-llm`에 올리면 `load_story_llm("feedback")`이 내려받아 씁니다.

```python
from huggingface_hub import HfApi
HfApi().upload_file(path_or_fileobj="feedback.pt", path_in_repo="feedback.pt", repo_id="JosephLee/modu-llm")
```

11월 5일에 만든 결과: 처음 보는 질문 24번 가운데 21번 모른다고 답함, 이야기 부탁 15번은 모두 이야기를 씀, 확인용 이야기 손실 0.645 → 0.657.